In [1]:
%load_ext autotime

time: 265 µs (started: 2024-04-22 14:30:43 +03:00)


In [2]:
import pandas as pd
import subprocess
from multiprocessing.pool import ThreadPool
from kubernetes import client, config
from dateutil import parser
import re

time: 471 ms (started: 2024-04-22 14:30:44 +03:00)


In [3]:
def call_ssh_command(command: str, host: str = "gateway.st"):
    return subprocess.run(
        ["ssh", host, command], 
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ).stdout

time: 477 µs (started: 2024-04-22 14:30:44 +03:00)


In [4]:
port_pattern = re.compile("--port(\s+)?=")

time: 428 µs (started: 2024-04-22 14:30:44 +03:00)


In [5]:
task6_pattern = re.compile("kubectl\s+exec\s+(((pod/)?lab0-jupyter-[a-z0-9_-]+)|(deployment/lab0-jupyter))\s+--\s+ls\s+[a-z-]+\s+\/home\/jovyan\/\.jupyter(\/\s+)?")

time: 713 µs (started: 2024-04-22 14:30:45 +03:00)


In [6]:
STUDENTS_LIST_DIR = "../../students/2024-Spring-RU"

time: 266 µs (started: 2024-04-22 14:30:46 +03:00)


In [7]:
students = pd.read_csv(f"{STUDENTS_LIST_DIR}/all_groups_data_uid.csv")

time: 4.96 ms (started: 2024-04-22 14:30:48 +03:00)


In [8]:
students.head(10)

,login,password,group,full_name,uid
0,tabbasov-410793,3j548it4SW,2024-Spring-RU,Abbasov Timur Agadzhanovich,10289
1,dandreev-369734,62Y3n11Iaz,2024-Spring-RU,Andreev Danil Evgenevich,10290
2,mantropova-370579,2zo5q3n8g0,2024-Spring-RU,Antropova Marina Mihajlovna,10184
3,nbarbara-410912,17zvGh838N,2024-Spring-RU,Barbara Nikita Sergeevich,10291
4,rbashirov-414146,b2G5s6V02q,2024-Spring-RU,Bashirov Roman Eduardovich,10292
5,lbeloglazov-410932,39a8TD14KN,2024-Spring-RU,Beloglazov Leonid Alekseevich,10293
6,mborodaj-410980,386EV1JWZ5,2024-Spring-RU,Borodaj Maksim Sergeevich,10294
7,vvetrov-414193,3IhHz7162j,2024-Spring-RU,Vetrov Vladislav Olegovich,10295
8,vvinnichenko-411068,0W47g3Xf5W,2024-Spring-RU,Vinnichenko Veronika Sergeevna,10296
9,mvitko-411071,k9u7B523nn,2024-Spring-RU,Vitko Mihail Aleksandrovich,10297


time: 9.66 ms (started: 2024-04-22 14:30:49 +03:00)


In [9]:
config.load_kube_config()
v1 = client.CoreV1Api()
apps_v1 = client.AppsV1Api()

time: 23.3 ms (started: 2024-04-22 14:30:52 +03:00)


In [10]:
def get_deployment(namespace: str, name: str = "lab0-jupyter"):
    return apps_v1.read_namespaced_deployment(name=name, namespace=namespace)

time: 422 µs (started: 2024-04-22 14:30:52 +03:00)


In [11]:
def get_service(namespace: str, name: str = "lab0-jupyter-service"):
    return v1.read_namespaced_service(name=name, namespace=namespace)

time: 322 µs (started: 2024-04-22 14:30:53 +03:00)


In [12]:
def get_config_map(namespace: str, name: str = "lab0-jupyter-cm"):
    return v1.read_namespaced_config_map(name=name, namespace=namespace)

time: 333 µs (started: 2024-04-22 14:30:53 +03:00)


In [13]:
def check_task1(deployment, service, report: dict):
    
    if deployment is not None:
        # ============== D ================
        report["1d"]["Status"] = "Completed"
        # ============== D ================
        
        
        # ============== A ================
        is_a_completed = False
        is_resources_specified = False
        is_limits_specified = False
        is_requests_specified = False
        is_limit_and_requests_equal = False
        is_guaranteed_qos = False
        is_required_resources = True

        resources = deployment.spec.template.spec.containers[0].resources
        if resources is not None:
            is_resources_specified = True

        if is_resources_specified:
            if resources.limits is not None:
                is_limits_specified = True

        if is_limits_specified:
            cpu_limit = resources.limits["cpu"]
            ram_limit = resources.limits["memory"]

        if is_resources_specified:
            if resources.requests is not None:
                is_requests_specified = True

        if is_requests_specified and is_limits_specified:
            cpu_limit = resources.limits["cpu"]
            ram_limit = resources.limits["memory"]
            cpu_requests = resources.limits["cpu"]
            ram_requests = resources.limits["memory"]
            if cpu_limit == cpu_requests and ram_limit == ram_requests:
                is_limit_and_requests_equal = True

        if is_resources_specified and is_limits_specified and (not is_requests_specified or is_limit_and_requests_equal):
            is_guaranteed_qos = True

        if is_guaranteed_qos:
            cpu_limit = resources.limits["cpu"]
            ram_limit = resources.limits["memory"]
            if (
                cpu_limit == "1" or \
                cpu_limit == "1000m"
               ) \
               and  \
               (
                ram_limit == "2048Mi" or \
                ram_limit == "2Gi" or \
                ram_limit == "2147483648" or \
                ram_limit == "2097152Ki"
               ):
                is_required_resources = True

        if is_guaranteed_qos and is_required_resources:
            is_a_completed = True
        
        if is_a_completed:
            report["1a"]["Status"] = "Completed"
        else:
            if is_guaranteed_qos and not is_required_resources:
                report["1a"]["Status"] = "Failed"
                report["1a"]["Reason"] = f"Pod's QoS is Guaranteed but the resources do not face the reqirements. Desired: \{cpu: 1, memory: 2Gi\}. Yours: \{cpu: {resources.limits['cpu']}, memory: {resources.limits['memory']}\}."
            elif not is_resources_specified:
                report["1a"]["Status"] = "Failed"
                report["1a"]["Reason"] = "You have't specified resources for your Jupyter-container."
            else:
                report["1a"]["Status"] = "Failed"
                report["1a"]["Reason"] = "Pod's QoS is not Guaranteed"
        # ============== A ================

        
        # ============== B ================
        is_b_completed = deployment.spec.replicas == 1
        
        if is_b_completed:
            report["1b"]["Status"] = "Completed"
        else:
            report["1b"]["Status"] = "Failed"
            report["1b"]["Reason"] = f"Desired replicas number: 1. Yours replicas number: {deployment.spec.replicas}"
        # ============== B ================

        
        # ============== C ================
        label_key = "jupyter"
        label_value = "lab0"
        is_c_completed = label_key in deployment.spec.template.metadata.labels \
                         and deployment.spec.template.metadata.labels[label_key] == label_value
        
        if is_c_completed:
            report["1c"]["Status"] = "Completed"
        else:
            report["1c"]["Status"] = "Failed"
            report["1c"]["Reason"] = "The required label hasn't been found in the template definition."
        # ============== C ================

        
    if service is not None:
        # ============== E ================
        report["1e"]["Status"] = "Completed"
        # ============== E ================
        
        
        # ============== F ================
        is_ports_number_satisfies = False
        is_endpoints_satisfies = False
        
        endpoints = v1.read_namespaced_endpoints(name=service.metadata.name, namespace=service.metadata.namespace)
        
        ports_count = len(service.spec.ports)
        
        endpoints_count = (
            0 if endpoints.subsets is None else
            0 if endpoints.subsets[0].addresses is None else
            len(endpoints.subsets[0].addresses)
        )
        
        if ports_count == 1:
            is_ports_number_satisfies = True
        
        if is_ports_number_satisfies:
            
            if endpoints_count == 1:
                is_endpoints_satisfies = True
        
        if is_endpoints_satisfies:
            report["1f"]["Status"] = "Completed"
        elif not is_ports_number_satisfies:
            report["1f"]["Status"] = "Failed"
            report["1f"]["Reason"] = f"Required ports number: 1. Your ports number: {ports_count}"
        else:
            report["1f"]["Status"] = "Failed"
            report["1f"]["Reason"] = f"Service should forward traffic to 1 pod. Your service is forwarding traffic into {endpoints_count} pods."
        # ============== F ================
        
        
        # ============== G ================
        if service.spec.type == "NodePort":
            report["1g"]["Status"] = "Completed"
        else:
            report["1g"]["Status"] = "Failed"
            report["1g"]["Reason"] = f"Service has to be NodePort type. Your type: {service.spec.type}"
        # ============== G ================


time: 2.98 ms (started: 2024-04-22 14:30:54 +03:00)


In [14]:
def check_task2(deployment, service, password: str, report: dict):
    
    if deployment is not None:
        # ============== A ================
        is_required_port = False
        
        args = deployment.spec.template.spec.containers[0].args
        port = None
        
        if args is None:
            args = []
        
        for arg in args:
            if "NotebookApp.port" in arg:
                try:
                    port = int(arg.split("=")[-1])
                except ValueError:
                    pass
        
        if "--port" in args:
            try:
                port = int(args[args.index("--port") + 1])
            except ValueError:
                pass
        else:
            for arg in args:
                if port_pattern.findall(arg):
                    _arg_splitted = arg.split("=")
                    if len(_arg_splitted) > 0:
                        try:
                            port = int(_arg_splitted[-1].strip())
                        except ValueError:
                            pass
        
        if port == 8282:
            is_required_port = True
        
        if is_required_port:
            report["2a"]["Status"] = "Completed"
        else:
            report["2a"]["Status"] = "Failed"
            report["2a"]["Reason"] = f"Required port: 8282. Your port: {port}"
        # ============== A ================
    
    
        # ============== B ================
        is_jupyter_token_corresponds_to_password = False
        
        for arg in args:
            if "NotebookApp.token" in arg:
                jupyter_token = arg.split("=")[-1].replace("'", "")
                if jupyter_token == password:
                    is_jupyter_token_corresponds_to_password = True
        
        if is_jupyter_token_corresponds_to_password:
            report["2b"]["Status"] = "Completed"
        else:
            report["2b"]["Status"] = "Failed"
            report["2b"]["Reason"] = "Jupyter token doesn't correspond to your gateway's password."
        # ============== B ================
    
    if service is not None:
        # ============== C ================
        is_svc_port_correct = False
        is_target_port_correct = False
        
        svc_port = service.spec.ports[0].port
        if svc_port == 80:
            is_svc_port_correct = True
        
        target_port = service.spec.ports[0].target_port
        if target_port == 8282:
            is_target_port_correct = True
        
        if is_svc_port_correct and is_target_port_correct:
            report["2c"]["Status"] = "Completed"
        elif not is_svc_port_correct:
            report["2c"]["Status"] = "Failed"
            report["2c"]["Reason"] = f"Service has to listen 80 port. Your Service listen: {svc_port}"
        else:
            report["2c"]["Status"] = "Failed"
            report["2c"]["Reason"] = f"Service has to forward traffic to 8282 port. Your Service is forwarding to {target_port} port."
        # ============== C ================
    
    if deployment is not None and service is not None:
        report["2d"]["Status"] = "Completed"
    elif deployment is None:
        report["2d"]["Status"] = "Failed"
        report["2d"]["Reason"] = "No Deployment found"
    else:
        report["2d"]["Status"] = "Failed"
        report["2d"]["Reason"] = "No Service found"


time: 2.79 ms (started: 2024-04-22 14:30:55 +03:00)


In [15]:
def check_task3(cm, report: dict):
    
    if cm is not None:
        # ============== A ================  
        report["3a"]["Status"] = "Completed"
        # ============== A ================
        
        
        # ============== B ================
        is_key_provided = False
        
        if "jupyter_notebook_config.py" in cm.data:
            report["3b"]["Status"] = "Completed"
            is_key_provided = True
        else:
            report["3b"]["Status"] = "Failed"
            report["3b"]["Reason"] = "Key 'jupyter_notebook_config.py' is not provided in configmap."
            
            report["3c"]["Status"] = "Failed"
            report["3c"]["Reason"] = "Key 'jupyter_notebook_config.py' is not provided in configmap."
        # ============== B ================        
        
        
        # ============== C ================
        if is_key_provided:
            is_content_correct = True
            
            content = list(map(lambda x: x.strip().replace(" ", ""), cm.data["jupyter_notebook_config.py"].split("\n")))
            content = list(filter(lambda x: len(x) > 0, content))
            
            # print(content)
            
            rows = ["c.NotebookApp.trust_xheaders=True", "c.NotebookApp.quit_button=False"]
            
            if len(content) != len(rows):
                is_content_correct = False
            
            correct_rows = 0
            for row in rows:
                # TODO: Should be refactored
                for c_row in content:
                    if row == c_row:
                        correct_rows += 1
                        break
            
            if correct_rows != len(rows):
                is_content_correct = False
            
            if is_content_correct:
                report["3c"]["Status"] = "Completed"
            else:
                report["3c"]["Status"] = "Failed"
                report["3c"]["Reason"] = f"Content of 'jupyter_notebook_config.py' is incorrect. Correct content: {rows}. Your content: {content}"
        # ============== C ================
    
    return report

time: 1.15 ms (started: 2024-04-22 14:30:55 +03:00)


In [16]:
def check_task4(deployment, report):
    
    jupyter_cfg_py = "jupyter_notebook_config.py"
    
    if deployment is not None:
        
        is_volume_exists = False
        is_volume_correct = False
        is_volume_mount_exists = False
        
        volumes = deployment.spec.template.spec.volumes
        volume_mounts = deployment.spec.template.spec.containers[0].volume_mounts
        
        if volumes is None:
            volumes = []
            
        if volume_mounts is None:
            volume_mounts = []
        
        cm_volume = None
        cm_volume_mount = None
        
        for volume in volumes:
            if volume.config_map is not None and volume.config_map.name == "lab0-jupyter-cm":
                cm_volume = volume
                is_volume_exists = True
                break
        
        
        if not is_volume_exists:
            for task in ["4a", "4b", "4c"]:
                report[task]["Status"] = "Failed"
                report[task]["Reason"] = "No ConfigMap volume found"
        
        if is_volume_exists:
            
            for vol_mount in volume_mounts:
                if vol_mount.name == cm_volume.name:
                    cm_volume_mount = vol_mount
                    is_volume_mount_exists = True
                    break
            
        
        if is_volume_exists and not is_volume_mount_exists:
            for task in ["4a", "4b", "4c"]:
                    report[task]["Status"] = "Failed"
                    report[task]["Reason"] = f"Your ConfigMap volume '{cm_volume.name}' is not mounted"
        
        if is_volume_mount_exists:
            
            if cm_volume_mount.read_only:
                report["4c"]["Status"] = "Completed"
            else:
                report["4c"]["Status"] = "Failed"
                report["4c"]["Reason"] = f"Your volume mount is not read only." 
            
            
            are_key_path_specified = False
            is_cm_key_correct = True
            
            items = cm_volume.config_map.items
            item = None if items is None else items[0]
            
            if item is not None and (item.key is not None or item.path is not None):
                are_key_path_specified = True
                if item.key != jupyter_cfg_py:
                    is_cm_key_correct = False
            
            if not is_cm_key_correct:
                for task in ["4a", "4b"]:
                    report[task]["Status"] = "Failed"
                    report[task]["Reason"] = f"Your ConfigMap volume '{cm_volume.name}' hasn't been set properly due to you have specified 'key' but wrong key name '{item.key}' is provided."
            else:
                if cm_volume_mount.mount_path == "/home/jovyan/.jupyter/jupyter_notebook_config.py":
                    report["4b"]["Status"] = "Completed"
                else:
                    report["4b"]["Status"] = "Failed"
                    report["4b"]["Reason"] = f"You have mounted your volume into wrong path. Desired: '/home/jovyan/.jupyter/jupyter_notebook_config.py'. Yours: '{cm_volume_mount.mount_path}'" 

                correct_subpath = jupyter_cfg_py if not are_key_path_specified else item.path
                if cm_volume_mount.sub_path == correct_subpath:
                    report["4a"]["Status"] = "Completed"
                else:
                    report["4a"]["Status"] = "Failed"
                    report["4a"]["Reason"] = f"You have used wrong ConfigMap paths to mount file."
        

time: 2.76 ms (started: 2024-04-22 14:30:56 +03:00)


In [17]:
def check_task5(login: str, report: dict):
    
    file_path = f"/home/{login}/lab0-jupyter.log"
    output = call_ssh_command(f"sudo cat {file_path}").split("\n")[:-1]
    
    if (len(output) > 0) and (file_path in output[0]) and ("No such file or directory" in output[0]):
        report["5"]["Status"] = "Failed"
        report["5"]["Reason"] = f"Log file not found."
    else:
        if len(output) < 10:
            report["5"]["Status"] = "Failed"
            report["5"]["Reason"] = f"Wrong log content"
        else:
           
            is_succeeded = True
            for row in output:
                try:
                    parser.parse(row.split(" ")[0].strip())
                except:
                    is_succeeded = False
                    break
            if is_succeeded:
                report["5"]["Status"] = "Completed"
            else:
                report["5"]["Status"] = "Failed"
                report["5"]["Reason"] = f"Wrong log content"

time: 817 µs (started: 2024-04-22 14:30:57 +03:00)


In [18]:
def check_task6(login: str, report: dict):
    
    file_path = f"/home/{login}/lab0-ls-lah.sh"
    
    output = call_ssh_command(f"sudo cat {file_path}").split("\n")
    
    if file_path in output[0] and "No such file or directory" in output[0]:
        report["6"]["Status"] = "Failed"
        report["6"]["Reason"] = f"Command file not found."
    else:
        # print(output)
        output = list(filter(lambda r: r.startswith("kubectl"), output))
        
        if len(output) < 1:
            report["6"]["Status"] = "Failed"
            report["6"]["Reason"] = "Kubectl command not found"
        else:
            command = output[0]
            if " -it " in command:
                report["6"]["Status"] = "Failed"
                report["6"]["Reason"] = "You shouldn't use interactive mode"
            else:
                if task6_pattern.match(command):
                    report["6"]["Status"] = "Completed"
                else:
                    report["6"]["Status"] = "Failed"
                    report["6"]["Reason"] = "Wrong command"

time: 805 µs (started: 2024-04-22 14:30:57 +03:00)


In [19]:
def check_all_tasks(login: str, password: str, check_only: list = None):
    
    deployment_name = "lab0-jupyter"
    service_name = "lab0-jupyter-service"
    cm_name = "lab0-jupyter-cm"
    
    report = {
        task: {
            "Status": None,
            "Reason": None
        } for task in [
            "1a", "1b", "1c", "1d", "1e", "1f", "1g",
            "2a", "2b", "2c", "2d",
            "3a", "3b", "3c",
            "4a", "4b", "4c",
            "5", "6"
        ]
    }
    
    deployment = None
    try:
        deployment = get_deployment(name=deployment_name, namespace=login)
    except client.rest.ApiException as e:
        if e.status == 404:
            for task in [
                "1a", "1b", "1c", "1d", "1f",
                "2a", "2b",
                "4a", "4b", "4c"
            ]:
                report[task]["Status"] = "Failed"
                report[task]["Reason"] = "No Deployment found"
    
    service = None
    try:
        service = get_service(name=service_name, namespace=login)
    except client.rest.ApiException as e:
        if e.status == 404:
            for task in [
                "1e", "1g",
                "2c"
            ]:
                report[task]["Status"] = "Failed"
                report[task]["Reason"] = "No Service found"
    
    cm = None
    try:
        cm = get_config_map(name=cm_name, namespace=login)
    except client.rest.ApiException as e:
        if e.status == 404:
            for task in [
                "3a", "3b", "3c"
            ]:
                report[task]["Status"] = "Failed"
                report[task]["Reason"] = "No ConfigMap found"
    
    check_task1(deployment=deployment, service=service, report=report)
    check_task2(deployment=deployment, service=service, password=password, report=report)
    check_task3(cm=cm, report=report)
    check_task4(deployment=deployment, report=report)
    check_task5(login=login, report=report)
    check_task6(login=login, report=report)
    
    return report

time: 2.08 ms (started: 2024-04-22 14:30:58 +03:00)


In [20]:
def check_user(login: str, password: str):
    result: dict = check_all_tasks(login=login, password=password)
        
    return [
        {
            "User": login,
            "Task": int(k[0]),
            "Sub-item": k[1] if len(k) > 1 else None,
            "Status": v["Status"],
            "Reason": v["Reason"]
        } for k, v in result.items()
    ]

time: 524 µs (started: 2024-04-22 14:30:59 +03:00)


In [21]:
def check_all_users(df: pd.DataFrame):
    results = list()
    for _, row in df.iterrows():
        print(f"Checking user {row['login']}...")
        results.extend(
            check_user(login=row["login"], password=row["password"])
        )
    return results

time: 462 µs (started: 2024-04-22 14:31:00 +03:00)


In [ ]:
# Проверить лабы у всех студентов или только у одного определённого
result = check_all_users(students[students["login"] == "spreobrazhenskij-412139"])

In [ ]:
student_results = pd.DataFrame(result).sort_values(by=["User", "Task", "Sub-item"], ascending=True)

In [24]:
student_results

,User,Task,Sub-item,Status,Reason
0,spreobrazhenskij-412139,1,a,Completed,None
1,spreobrazhenskij-412139,1,b,Completed,None
2,spreobrazhenskij-412139,1,c,Completed,None
3,spreobrazhenskij-412139,1,d,Completed,None
4,spreobrazhenskij-412139,1,e,Completed,None
5,spreobrazhenskij-412139,1,f,Completed,None
6,spreobrazhenskij-412139,1,g,Completed,None
7,spreobrazhenskij-412139,2,a,Completed,None
8,spreobrazhenskij-412139,2,b,Completed,None
9,spreobrazhenskij-412139,2,c,Completed,None


time: 9.09 ms (started: 2024-04-22 14:31:31 +03:00)
